# Week 1 · Notebook 01: Real Data from Kepler and TESS
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/01_download_and_plot.ipynb)

**Goal:** download a real light curve of a real planet-hosting star from NASA's archive (MAST), understand
every column in it, and make your first plots. **Time:** 2–3 hours.

### 🎯 By the end of this notebook you can
1. explain how a telescope image becomes a light curve;
2. search the MAST archive and download Kepler or TESS data;
3. explain SAP vs PDCSAP flux, quality flags, cadence, and the BKJD/BTJD time systems;
4. find data gaps and estimate the noise of a light curve;
5. predict, *before* searching, whether a known planet should be detectable (expected SNR);
6. pick and document your own target star.

### 🧭 Roadmap
| § | Topic |
|---|---|
| 0 | From photons to light curves |
| 1 | Search MAST |
| 2 | Look inside one file: columns, time, flux units, quality flags |
| 3 | SAP vs PDCSAP |
| 4 | Our pipeline's format, gaps and noise |
| 5 | Zoom, bin, and a sneak peek at the planet |
| 6 | The star, and the expected SNR |
| 7 | Your own target |

**Background reading (30 min, before you start):**
- Lightkurve tutorial: [Searching & downloading data](https://lightkurve.github.io/lightkurve/tutorials/1-getting-started/searching-for-data-products.html)
- HEASARC: [Understanding LightCurve objects](https://heasarc.gsfc.nasa.gov/docs/tess/LightCurve-object-Tutorial.html)
<details><summary><b>📎 How to read this notebook (click to open)</b></summary>

| Symbol | Meaning |
|---|---|
| 🔮 **Predict** | Before running the next cell, write down what you expect. Being wrong is how you learn. |
| 👀 **What to look for** | How to read the output or plot you just made. |
| ✏️ **Your turn** | An exercise. 🟢 everyone · 🟡 stretch · 🔴 challenge. |
| 💡 / ✅ | Click to reveal a hint or a worked solution. Try first, then peek. |
| ⚠️ **Pitfall** | A mistake that real researchers make. |
| 🧠 **Concept check** | Short quiz. Answers are hidden under each question. |
| `check(...)` | Self-check: prints ✅ when your exercise answer is right. |

Run cells **top to bottom** with **Shift+Enter**. If things get weird: *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).
</details>

In [ ]:
# ⚙️ SETUP: run this cell first, every time you open the notebook. Works in Google Colab and locally.
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})

def check(name, value, expected, rtol=0.02):
    """Self-check for the exercises. Prints ✅, ❌, or ⏳ (not attempted yet)."""
    if value is None:
        print(f"⏳ {name}: not attempted yet (replace None with your answer)")
        return
    try:
        ok = bool(np.allclose(value, expected, rtol=rtol))
    except Exception:
        ok = value == expected
    print(f"✅ {name}: correct!" if ok else f"❌ {name}: not quite (you gave {value!r}). Check the hint.")

print("ready, repo root:", ROOT)

## 0. From photons to light curves
```
 star light ──► telescope ──► CCD image ──► add up the star's pixels ──► one brightness value
                                (every 30 min for Kepler long cadence; 2 min / 200 s for TESS)
                                                │
                     repeat for months ─────────┘──► LIGHT CURVE = brightness vs time
                                                         │
             NASA pipeline removes spacecraft effects ───┘──► PDCSAP flux  ──► archived at MAST
```
- **Kepler** (2009–2013) stared at one patch of sky for 4 years, in **quarters** of ~90 days.
- **TESS** (2018–) scans almost the whole sky in **sectors** of ~27 days. Brighter stars, shorter coverage.
- **MAST** (Mikulski Archive for Space Telescopes) stores all of it. We talk to MAST through the
  `lightkurve` package.

## 1. Search MAST
`lk.search_lightcurve` asks MAST which data exist for a target. **Nothing downloads yet.**
Our shared target is **Kepler-10**, host of Kepler-10 b, the first rocky planet Kepler confirmed
(Batalha et al. 2011).

In [ ]:
import lightkurve as lk
TARGET = "Kepler-10"          # later try: "pi Men", "TOI-270", "Kepler-8", "WASP-18"
search = lk.search_lightcurve(TARGET, mission="Kepler", author="Kepler", exptime="long")
print(len(search), "light curve files found")
search

👀 **What to look for** in the table:
- **mission**: which quarter (Kepler) or sector (TESS) each row is;
- **author**: who made the light curve (`Kepler` / `SPOC` are the official NASA pipelines; others are community products);
- **exptime**: 1800 s = *long cadence* (one point per 30 min); 60 s = *short cadence*;
- **distance**: arcseconds from the position you searched. Should be ~0 for your star.

Kepler-10 has data in almost every quarter. We'll take two: **quarters 3 and 4**. Quarter 4 is short (~3 weeks)
for this star: in January 2010 one of Kepler's CCD modules failed, and Kepler-10 fell on it. Real data have stories like this.

In [ ]:
lc_file = lk.search_lightcurve(TARGET, mission="Kepler", author="Kepler",
                               exptime="long", quarter=3).download()
print(type(lc_file).__name__, "with", len(lc_file), "rows")
print("columns:", lc_file.colnames[:12], "...")

## 2. What is inside one file?
The file is a **FITS** table (the standard astronomy format): a header with metadata, and a table with one
row per measurement. Let's read some of the header: what the spacecraft knew about the star.

In [ ]:
for key in ["OBJECT", "KEPLERID", "QUARTER", "KEPMAG", "TEFF", "LOGG", "RADIUS"]:
    print(f"{key:9s} = {lc_file.meta.get(key)}")

`KEPMAG` is the brightness in the Kepler band (smaller = brighter; 11 is bright for Kepler), `TEFF` the
temperature in K, `RADIUS` a catalogue star radius in R☉. ⚠️ **Pitfall:** header radii can be old; for science
use the NASA Exoplanet Archive (Section 6).

### Time
Kepler times are in **BKJD** = BJD − 2454833 days; TESS uses **BTJD** = BJD − 2457000. BJD is the time
at the centre of mass of the Solar System, so a star's clock doesn't wobble as Earth/Kepler orbit the Sun.

In [ ]:
t = lc_file.time
print("time format:", t.format)
print("first time stamp:", t[0].value, "BKJD  =", t[0].jd, "BJD  =", t[0].iso, "(calendar date)")
print("cadence:", np.median(np.diff(t.value)) * 24 * 60, "minutes")

### Flux, units and quality flags
Flux comes in **electrons per second** collected by the CCD. Absolute values don't matter for transits:
we care about *fractional* changes, so we **normalise** (divide by the median) to get flux ≈ 1.

A `———` in the printout is a **masked** value: a cadence with no valid measurement.

The **QUALITY** column is a set of bit flags marking bad moments (thruster firings, cosmic rays, the spacecraft
pointing at Earth…). `lightkurve` already removed the worst ones by default.

In [ ]:
print("flux unit:", lc_file.flux.unit)
print("first fluxes:", lc_file.flux[:3])
print("missing (NaN) fluxes:", np.isnan(lc_file.flux.unmasked.value).sum(), "of", len(lc_file))
print("rows still carrying a (mild) quality flag:", np.sum(lc_file.quality.value != 0))

## 3. SAP vs PDCSAP flux
- **SAP_FLUX**: Simple Aperture Photometry. Just add up the pixels. Includes spacecraft systematics
  (slow drifts, jumps, focus changes).
- **PDCSAP_FLUX**: after NASA's *Presearch Data Conditioning* removed most of those instrumental trends,
  using what all the other stars on the same detector did.

🔮 **Predict:** which of the two panels will look flatter?

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
lc_file.select_flux("sap_flux").normalize().plot(ax=ax[0], label="SAP", color="k")
lc_file.select_flux("pdcsap_flux").normalize().plot(ax=ax[1], label="PDCSAP", color="C1")
ax[0].set_title("Kepler-10, quarter 3: raw (SAP) vs corrected (PDCSAP)")
plt.tight_layout(); plt.show()

👀 SAP shows a long drift that has nothing to do with the star. PDCSAP removed it. **Use PDCSAP for
transit searches.** (Advanced: PDC can occasionally distort very long transits or real stellar signals.
For your target, always look at both.)

## 4. Into our own pipeline's format
Our package `transitpipe` stores a light curve as three plain NumPy arrays (`time`, `flux`, `flux_err`)
plus a `meta` dictionary. Seeing raw numbers removes the magic. `get_lightcurve` downloads, picks PDCSAP,
drops NaNs, stitches quarters together and normalises each one.

In [ ]:
from transitpipe import io, detrend, utils
lc = io.get_lightcurve("Kepler-10", mission="Kepler", exptime="long", quarter=[3, 4])
print(len(lc), "points")
print("time  :", lc.time[:4], "(BKJD days)")
print("flux  :", lc.flux[:4])
print("errors:", lc.flux_err[:4])

# Save a copy so later notebooks work offline (data/ is git-ignored: never commit big data files)
os.makedirs(os.path.join(ROOT, "data"), exist_ok=True)
io.to_csv(lc, os.path.join(ROOT, "data", "kepler10_q3q4.csv"))

### Gaps
Spacecraft stop observing: data downloads, safe modes, the gap between quarters. Gaps matter: smoothing
*across* a gap invents data, and a transit falling in a gap is simply lost. Let's find them with
`np.diff` (differences between neighbouring times) and a mask.

In [ ]:
dt = np.diff(lc.time)
print(f"typical spacing: {np.median(dt) * 24 * 60:.1f} min")
gap_idx = np.where(dt > 0.5)[0]           # indices where the next point is > 0.5 d later
for i in gap_idx:
    print(f"gap from {lc.time[i]:.2f} to {lc.time[i + 1]:.2f} BKJD ({dt[i]:.2f} d)")
total = lc.time[-1] - lc.time[0]
print(f"time span {total:.1f} d, of which {dt[gap_idx].sum():.1f} d is in big gaps")

### Noise
How noisy is each point? `estimate_noise_ppm` uses the MAD (median absolute deviation), a robust cousin of
the standard deviation that ignores outliers. Before detrending it also includes stellar variability, so it is
an *upper limit*.

In [ ]:
print(f"point-to-point scatter ≈ {detrend.estimate_noise_ppm(lc):.0f} ppm (before detrending)")
print(f"median quoted error    ≈ {np.median(lc.flux_err) * 1e6:.0f} ppm")

## 5. Plot it: the whole thing, a zoom, and binned
🔮 **Predict:** Kepler-10 b orbits every 0.84 days with a ~150 ppm dip. Will you see transits by eye in the
zoom panel?

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6.5))
ax[0].plot(lc.time, lc.flux, ".", ms=1, color="k")
ax[0].set_xlabel("Time [BKJD days]"); ax[0].set_ylabel("Normalised PDCSAP flux")
ax[0].set_title("Kepler-10, quarters 3–4")

zoom = (lc.time > lc.time[0] + 10) & (lc.time < lc.time[0] + 13)        # a mask: 3 days
ax[1].plot(lc.time[zoom], (lc.flux[zoom] - 1) * 1e6, ".", color="k", label="30-min points")
xb, yb = utils.bin_series(lc.time[zoom], lc.flux[zoom], bins=36)        # 2-hour averages
ax[1].plot(xb, (yb - 1) * 1e6, "o-", color="C1", label="2-hour averages")
ax[1].set_xlabel("Time [BKJD days]"); ax[1].set_ylabel("Flux − 1 [ppm]")
ax[1].set_title("Zoom on 3 days"); ax[1].legend()
plt.tight_layout(); plt.show()

👀 No, not by eye. The point-to-point noise is comparable to the dip, and the star itself wanders.

### A sneak peek (cheating, for motivation)
Next week you'll **find** the period blindly. Today, let's *cheat* and use the published ephemeris
(period and a mid-transit time) to fold the data, so you can see that the planet is in there.

In [ ]:
P_pub, t0_pub = 0.8374907, 131.5749          # published period [d] and mid-transit time [BKJD]
phase_hours = utils.phase_fold(lc.time, P_pub, t0_pub) * P_pub * 24
near = np.abs(phase_hours) < 5
xb, yb = utils.bin_series(phase_hours[near], lc.flux[near], bins=40)
plt.figure(figsize=(7, 4))
plt.plot(phase_hours[near], (lc.flux[near] - 1) * 1e6, ".", ms=1, color="0.7", label="data")
plt.plot(xb, (yb - 1) * 1e6, "o", color="C3", label="binned")
plt.ylim(-600, 400); plt.xlabel("Hours from mid-transit"); plt.ylabel("Flux − 1 [ppm]")
plt.title("Kepler-10 b, folded on the published period"); plt.legend(); plt.show()

👀 There it is: a ~1.8-hour, ~150 ppm dip, built from ~130 transits stacked on top of each other.
(The stellar wiggles haven't been removed yet; next week's detrending will make this much cleaner.)

## 6. Look up the star, and predict the SNR before searching
The planet's radius comes from $R_p = \sqrt{\delta}\,R_\star$, so the **star's radius** matters as much as the
depth. Look these up now and record them in your repo's README:
- [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) → search *Kepler-10* → stellar radius,
  mass, and the published planet radius (your Week 3 answer key!)
- For TESS stars: [ExoFOP-TESS](https://exofop.ipac.caltech.edu/tess/) (TIC catalogue values)

In [ ]:
STAR = {"name": "Kepler-10", "rstar": 1.065, "rstar_err": 0.009,     # R_sun (Fogtmann-Schulz+2014)
        "mstar": 0.910, "mstar_err": 0.021}                           # M_sun
PLANET_PUB = {"period": 0.8374907, "rp_rearth": 1.47}                 # published values

depth_exp = (PLANET_PUB["rp_rearth"] / (STAR["rstar"] * utils.R_SUN_IN_REARTH)) ** 2
dur_exp = utils.expected_transit_duration(PLANET_PUB["period"], STAR["rstar"], STAR["mstar"])
n_transits = (lc.time[-1] - lc.time[0]) / PLANET_PUB["period"]
cadence = np.median(np.diff(lc.time))
n_in = n_transits * dur_exp / cadence
noise = detrend.estimate_noise_ppm(lc) * 1e-6
print(f"expected depth     : {depth_exp * 1e6:.0f} ppm")
print(f"expected duration  : {dur_exp * 24:.1f} h")
print(f"in-transit points  : ~{n_in:.0f} (≈ {n_transits:.0f} transits)")
print(f"expected SNR       : ~{depth_exp / noise * np.sqrt(n_in):.0f}  (detectable if ≳ 7)")

👀 Even with the noise estimate inflated by stellar variability, the expected SNR is far above 7, so BLS
should find Kepler-10 b easily next week. **Do this calculation for your own target too.** If the expected
SNR is below ~10, pick a different star (or more quarters/sectors).

## 7. ✏️ Your turn: pick YOUR target
Everyone does Kepler-10 first (shared answer key). Then each person picks their **own** star so every repo is
unique. Good first choices: a confirmed planet with depth **> 500 ppm** and period **< 10 d**.
Browse the [Exoplanet Archive table](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TblView/nph-tblView?app=ExoTbls&config=PSCompPars)
(filter `disc_facility` = Kepler or TESS) or use the target list in the project guide (Section 7).

**TESS differences:** use `mission="TESS"`, `author="SPOC"`, `exptime=120` (2-min cadence), and
`sector=[...]` instead of `quarter`. Time is in BTJD.

1. 🟢 Search, download and plot your target (both SAP and PDCSAP).
2. 🟢 Record star radius/mass (with errors and source) and the published period/radius in your README.
3. 🟡 Compute the expected SNR as in Section 6.
4. 🔴 Does your star vary (spots rotating in and out of view)? Estimate the rotation period by eye, then with
   `lc_file.normalize().to_periodogram().plot()` (a Lomb–Scargle periodogram).

<details><summary>💡 Example for a TESS star (π Mensae)</summary>

```python
my = io.get_lightcurve("pi Men", mission="TESS", exptime="120", sector=[1])
plt.plot(my.time, my.flux, ".", ms=1); plt.xlabel("Time [BTJD]"); plt.ylabel("Flux"); plt.show()
```
</details>

In [ ]:
# ✏️ YOUR CODE HERE
MY_TARGET = "pi Men"      # change me
my_search = lk.search_lightcurve(MY_TARGET, mission="TESS", author="SPOC")
my_search

## 🧠 Concept check
**Q1.** Why use PDCSAP rather than SAP flux for a transit search?
<details><summary>Answer</summary>PDCSAP has most spacecraft systematics removed, so remaining variations are the star (and planets), not the instrument.</details>

**Q2.** A Kepler time stamp reads 300.0 BKJD. What is it in BJD?
<details><summary>Answer</summary>300.0 + 2454833 = 2455133.0 BJD.</details>

**Q3.** A star has 800 ppm point-to-point noise. Can a 200 ppm transit be found? What would you need?
<details><summary>Answer</summary>Yes, if enough in-transit points are averaged: SNR = 0.25·√N ≥ 7 needs N ≳ 800 in-transit points, i.e. many transits (more data) or faster cadence.</details>

**Q4.** Why should you not commit downloaded FITS files to GitHub?
<details><summary>Answer</summary>They are big, anyone can re-download them from MAST, and git keeps every version forever. Commit the code that downloads them instead.</details>

## 📌 Key takeaways
- MAST holds the data; `lightkurve` searches and downloads it. Kepler = quarters, TESS = sectors.
- Use PDCSAP flux, normalised. Watch for gaps and quality flags.
- Times are BKJD (Kepler) or BTJD (TESS): BJD minus a constant.
- Predict the SNR *before* searching: depth, duration, noise and the number of transits.

## 📖 Glossary
**cadence**: time between measurements · **quarter / sector**: Kepler / TESS observing blocks ·
**SAP / PDCSAP**: raw / systematics-corrected flux · **BKJD / BTJD**: mission time systems ·
**MAD**: median absolute deviation (robust scatter) · **ephemeris**: period + mid-transit time.

### ✅ Week 1 Checkpoint
- [ ] Your own GitHub repo with Notebooks 00 and 01 committed, and a README stating your target
- [ ] A labelled raw light-curve plot of your target
- [ ] A table (star radius, mass, published P and Rp, with sources) and your expected SNR

**➡️ Next:** Notebook 02. Flatten the star's wiggles and *find* the planet without cheating.

In [ ]:
# Your notes and extra experiments